In [ ]:
# ## PHẦN 4 - CHIA TRAIN/TEST THEO THỜI GIAN
# Dùng 80% dữ liệu đầu theo thời gian để train và 20% cuối để test,
# không shuffle; cùng một cách chia được dùng cho cả năm mô hình.

TEST_FRACTION = 0.20
split_data = {}
split_summary_rows = []

for continent in CONTINENTS:
    continent_data = (
        df_features.loc[df_features["Continent"] == continent]
        .sort_values("Date")
        .reset_index(drop=True)
    )
    if len(continent_data) < 2:
        raise ValueError(f"Không đủ dòng để chia train/test cho {continent}.")

    test_count = max(1, int(np.ceil(len(continent_data) * TEST_FRACTION)))
    split_index = len(continent_data) - test_count
    train_data = continent_data.iloc[:split_index].copy()
    test_data = continent_data.iloc[split_index:].copy()
    if train_data.empty or test_data.empty:
        raise ValueError(f"Tập train/test rỗng cho {continent}.")

    split_data[continent] = {
        "train": train_data,
        "test": test_data,
    }
    split_summary_rows.append(
        {
            "Continent": continent,
            "Train rows": len(train_data),
            "Train start": train_data["Date"].min().date(),
            "Train end": train_data["Date"].max().date(),
            "Test rows": len(test_data),
            "Test start": test_data["Date"].min().date(),
            "Test end": test_data["Date"].max().date(),
        }
    )

split_summary = pd.DataFrame(split_summary_rows)
display(split_summary)

In [ ]:
# ## PHẦN 5 - HUẤN LUYỆN 5 MÔ HÌNH
# Linear Regression dự đoán trực tiếp New_Confirmed. Bốn mô hình còn lại
# dự đoán Diff_1 rồi cộng Lag_1 để đổi dự báo về số ca mới.

def make_models() -> Dict[str, Tuple[str, object]]:
    """Khởi tạo cùng cấu hình mô hình cho từng châu lục."""
    return {
        "Linear Regression (Level)": (
            "level",
            LinearRegression(),
        ),
        "Ridge Regression (Diff)": (
            "diff",
            Ridge(alpha=1.0),
        ),
        "Random Forest Regressor (Diff)": (
            "diff",
            RandomForestRegressor(
                n_estimators=200,
                max_depth=6,
                min_samples_leaf=2,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
        "Gradient Boosting Regressor (Diff)": (
            "diff",
            GradientBoostingRegressor(
                n_estimators=100,
                learning_rate=0.05,
                max_depth=2,
                random_state=RANDOM_STATE,
            ),
        ),
        "Decision Tree Regressor (Diff)": (
            "diff",
            DecisionTreeRegressor(
                max_depth=5,
                min_samples_leaf=3,
                random_state=RANDOM_STATE,
            ),
        ),
    }


fitted_models = {}
prediction_parts = []
model_settings = {
    "Linear Regression (Level)": "default",
    "Ridge Regression (Diff)": "alpha=1.0",
    "Random Forest Regressor (Diff)": (
        "n_estimators=200, max_depth=6, min_samples_leaf=2"
    ),
    "Gradient Boosting Regressor (Diff)": (
        "n_estimators=100, learning_rate=0.05, max_depth=2"
    ),
    "Decision Tree Regressor (Diff)": "max_depth=5, min_samples_leaf=3",
}

for continent in CONTINENTS:
    train_data = split_data[continent]["train"]
    test_data = split_data[continent]["test"]
    x_train = train_data[FEATURE_COLUMNS]
    x_test = test_data[FEATURE_COLUMNS]

    continent_models = {}
    for model_name, (target_mode, model) in make_models().items():
        target_column = "New_Confirmed" if target_mode == "level" else "Diff_1"
        model.fit(x_train, train_data[target_column])

        predicted_target = model.predict(x_test)
        if target_mode == "diff":
            predicted_cases = predicted_target + test_data["Lag_1"].to_numpy()
        else:
            predicted_cases = predicted_target

        continent_models[model_name] = {
            "model": model,
            "target_mode": target_mode,
        }
        prediction_parts.append(
            pd.DataFrame(
                {
                    "Date": test_data["Date"].to_numpy(),
                    "Continent": continent,
                    "Model": model_name,
                    "Actual": test_data["New_Confirmed"].to_numpy(),
                    "Predicted": predicted_cases,
                }
            )
        )
    fitted_models[continent] = continent_models

predictions = pd.concat(prediction_parts, ignore_index=True)
print("Số dòng dự báo:", len(predictions))
display(predictions.head())
print("Tham số mô hình:")
display(
    pd.DataFrame(
        [
            {"Model": name, "Parameters": settings}
            for name, settings in model_settings.items()
        ]
    )
)